## Configuration

In [ ]:
from pathlib import Path
import tomllib
from utils.utils import *
import json

PATH_EEG = Path("../EEG/EEG/results/")
FILENAME = 'Case_02_ExG.csv'
MARKER_FILE = 'Case_02_Marker.csv'
CONFIG_PATH = "../EEG/EEG/config/config.toml"

with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

BANDS = config['BANDS']
SEGMENT_DEF = config['SEGMENT_DEF']
RANDOM_SEED = config['RANDOM_SEED']
DURATION = config['DURATION']

SFREQ = config['SFREQ']
L_FREQ = config['L_FREQ']
H_FREQ = config['H_FREQ']

## Raw Data

In [ ]:
raw_full, df_full, df_markers_full = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

raws, labels = extract_segments(raw_full, df_full, df_markers_full, SEGMENT_DEF)

In [ ]:
plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha', bands=BANDS)
plot_topomap_comparison(raws, labels, band_name='Beta', bands=BANDS)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta', bands=BANDS)
plot_topomap_comparison(raws, labels, band_name='Theta', bands=BANDS)

## Preprocessing

In [ ]:
import mne
from mne.preprocessing import ICA
raw, df, df_markers = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

# ICA
# n_components=15 (16 Channels, ICA finding max n_channels - 1 or same number)
ica = ICA(n_components=15, max_iter='auto', random_state=RANDOM_SEED)

print(">>> ICA (Might take a while)...")
ica.fit(raw)

print(">>> Showing Components...")
ica.plot_components()
plt.show()

ica.plot_sources(raw, show_scrollbars=True)
plt.show()

In [ ]:
# ica.plot_properties(raw, picks=range(15))

In [ ]:
exclude_indices = [0, 1, 7, 12] 

print(f">>> Removing ingredients: {exclude_indices}")
ica.exclude = exclude_indices

raw_clean = raw.copy()
ica.apply(raw_clean)

print(">>> Cleaned data.")

target_chan = 'F3' if 'F3' in raw.ch_names else raw.ch_names[0]

print(f"Comparison on the channel {target_chan}...")
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(raw.get_data(picks=target_chan)[0, 1000:2500], color='red', alpha=0.5, label='Before cleaning (Original)')
ax.plot(raw_clean.get_data(picks=target_chan)[0, 1000:2500], color='black', label='After cleaning (Clean)')
ax.set_title(f"Artifact Removal Effect (Eyes and Muscles) - Channel {target_chan}")
ax.legend()
plt.show()

In [ ]:
raws, labels = extract_segments(raw_clean, df_full, df_markers_full, SEGMENT_DEF)

plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha', bands=BANDS)
plot_topomap_comparison(raws, labels, band_name='Beta', bands=BANDS)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta', bands=BANDS)
plot_topomap_comparison(raws, labels, band_name='Gamma', bands=BANDS)

## Podsumowanie

### Alpha
 **Faza 1** Relaks, nie trzeba "przetwarzać" treści.

 **Faza 2** Relaks, ale nie taki jak w Fazie 1.
 
 **Faza 3** Faza ze scrollowanie, sygnał alpha jest słabszy niż przy fazie bez scrollowania, prawdopodobnie, dlatego, że mózg zaczyna analizować treść i oceniać, czy jest warta poświęcenia na nią zasobów.
 
 **Faza 4** Faza ze scrollowanie, sygnał alpha jest słabszy niż przy fazie bez scrollowania, prawdopodobnie, dlatego, że mózg zaczyna analizować treść i oceniać, czy jest warta poświęcenia na nią zasobów, plus fakt, że z tej przetwarza się informacje "wartościowe", nie tylko rozrywkowe.

### Beta
  Uczestnik praworęczny.
 
 **Faza 1** Przez szybki montaż mózg był zalewany "treścią", np. szybkim montażem
 
 **Faza 2** Treści smart były spokojniejsze przez co kanał O2 nie był tak bombardowany.
 
 **Faza 3** Większy "okrąg" przy C3 sugeruje *"zombie scrolling"*, szukał dopaminy, bądź pomijał kontent.
 
 **Faza 4** Mniejszy "okrąg" przy C3 sugeruje, że uczestnik nie *"zombie scrollował"*, ale przykładał większą uwagę do treści

 Ważna uwaga, płat czołowy w paśmie beta odpowiada za "twarde myślenie", fakt, że jest niebieski świadczy o tym, że mózg chłonie informacje, a nie je przetwarza.
 Może to oznaczać, że nawet treści edukacyjne mogą niebyć zbyt dużym wyzwaniem dla mózgu.

#### Płaty Ciemieniowe (P3, P4, P5, P6 - Orientacja i Uwaga)
 Widać wpływ scrollowania na uwagę.
Kiedy uczestnik scrolluje mądre treści, jego prawa półkula ciemieniowa (P4, P6) się zapala. Odpowiada ona za uwagę przestrzenną ("Gdzie patrzeć na ekranie?") i nawigację. Przy Brainrocie (F3) też są czerwone, ale przy Smart (F4) widać wyraźną różnicę względem braku scrollowania. Aktywne szukanie informacji angażuje płat ciemieniowy.

Mózg w fazie Smart Scroll (F4) działał najbardziej symetrycznie i efektywnie w obszarach wzrokowych i uwagowych, podczas gdy Brainrot wywoływał chaos (asymetrie, większe napięcie ruchowe).

Mimo różnych treści:
- Mózg najuważniej przetwarzał obraz tylko w fazie 4 (Oz)
- Faza 3 (Brainrot Scroll). Wielka plama na C3 sugeruje "nerwowy palec" i mniej precyzyjne sterowanie uwagą. Skakanie i szukanie dopaminy.
- Przód głowy (Czoło) w każdej fazie był chłodny (niebieski). To znaczy, że mimo różnic, żadna z tych czynności nie była stresująca ani wybitnie trudna intelektualnie. Uczestnik był w strefie komfortu.

### Delta

Kora mózgowa pracuje na zwolnionych obrotach widać to po O2, P4 i P6. Onzacza to, wpatrywanie się lub gapienie. Mózg nie analizuje aktywnie obrazu, lecz go chłonie jak gąbka.
Brainrot wywołuje przeładowanie sensoryczne, kora wzrokowa (O2) jest tak zbombardowana bodźcami, że wchodzi w specyficzny stan rezonansu/obronny (Delta), żeby to przetrwać.

Uczestnik może nieświadomie napinać jeden konkretny mięsień karku (z prawej strony) przy brainrocie, np. lekko wysuwając głowę lub sztywniejąc ("zastyganie" w bezruchu). Smart content nie wywołuje tego "zastygnięcia".

W Fazie 4 jest mniej fal Delta, mózg wychodzi z transu. Scrollowanie mądrych treści i czytanie ich wymaga, by kora wzrokowa się obudziła. Fale Delta znikają (robi się jaśniej), bo mózg zaczyna przetwarzać detale, a nie tylko chłonąć kolorowe obrazki.


### Gamma

Mimo, że scrolowanie było wyłączone to C3 jest ciemno czerwone, to oznacza statyczne napięcie mięśniowe. Trzymanie telefonu w gotowości, albo kora ruchowa była "podminowana". Mózg "chciał" scrollować (dopamina), mimo że tego nie robił. To objaw niepokoju ruchowego.

W fazie 3 C3 jest czerowny przez scrollowanie.

W fazie 4, kiedy mózg zajął się analizą treści (Smart), napięcie w ręce spadło. Nadmierne pobudzenie (niepokój) zniknęło, ustępując miejsca celowemu działaniu.

#### Płat Ciemieniowy (P3, P4, P5, P6) – "Pustka" vs "Myślenie"
Płaty ciemieniowe (P) w Gammie odpowiadają za łączenie faktów (asocjację).
W faza 1 i 3 wszystkie kanały P (P3, P5, Pz, P4, P6) są JASNO NIEBIESKIE.
Informacja wpada do oka (O1/O2), ręka się rusza (C3), ale środek mózgu (P) jest wyłączony. brak analizy tego, co widać. Obraz przelatuje przez głowę bez zatrzymywania się. To definicja "zombie-mode".

W fazie 4 (Smart Scroll): P3, P5 (lewa strona) są JASNO CZERWONE/BIAŁE, a Pz, P4, P6 (prawa strona) są JASNO CZERWONE. Obszary, które w Brainrocie spały (były niebieskie), teraz świecą na czerwono. Mózg aktywnie łączy obraz z wiedzą.

#### Kora Wzrokowa (O1, O2)
W fazie 4 O1 jest CIEMNO CZERWONY (Najsilniejszy punkt), O2 jest jasno czerwony.
Lewa półkula odpowiada za język, tekst i analizę sekwencyjną.
Oglądając "Smart" treści, prawdopodobnie czytałeś napisy, słuchałeś słów i analizowałeś logiczny ciąg zdarzeń. Lewa kora wzrokowa (O1) pracowała na najwyższych obrotach, przekazując dane do lewego płata ciemieniowego (P3, P5), który też się "zapalił".

EEG udowadnia, że Brainrot to nie relaks. To stan, w którym kora ruchowa jest spięta (C3 Gamma), kora wzrokowa jest w transie (O2 Delta), a ośrodki myślenia są wyłączone (P Niebieskie). Z kolei Smart Content – mimo że wymaga wysiłku (aktywne P i O1 w Gammie) – paradoksalnie uspokaja motorykę (spadek napięcia na C3) i wybudza mózg z letargu.

## Spektogram

In [ ]:
# Alpha (fatigue/relax)
plot_band_power_over_time(raws, labels, band=(8, 12), band_name='Alpha')
# Beta (Focus)
plot_band_power_over_time(raws, labels, band=(13, 30), band_name='Beta')

Około 82 sekundy w Fazie smart scroll widać przecięcie fal alfa i beta, to moment pełnej aktywacji.
Widać to też około 20 sekuny w tej samej fazie.

In [ ]:
# Emotion
calculate_asymmetry(raws, labels, ch_left='F3', ch_right='F4')

# According to Heller's model, parietal asymmetry is responsible for physiological arousal
calculate_asymmetry(raws, labels, ch_left='P3', ch_right='P4')

# Image processing method.
calculate_asymmetry(raws, labels, ch_left='O1', ch_right='O2')

# Motor
calculate_asymmetry(raws, labels, ch_left='C3', ch_right='C4')

### Podsumowanie

#### Kora Czołowa (F4 vs F3) – Motywacja i Emocje
Ten wykres pokazuje zjawisko FAA (Frontal Alpha Asymmetry). Wskaźnik powyżej zera oznacza motywację dążenia (zaangażowanie, pozytywny afekt, ciekawość), a poniżej zera – reakcję wycofania (znużenie, dystans, negatywny afekt).

Stan ogólny "Brainrot": Widać potężny spadek poniżej zera. Mózg uczestnika "wyłącza się" emocjonalnie. Wykazuje znużenie i awersję do szumu informacyjnego. Wpada w swego rodzaju letarg poznawczy.

Stan ogólny "Smart": Wskaźnik jest lekko dodatni. Uczestnik przetwarza treści w sposób naturalny i ze stabilnym nastawieniem.

Sam moment Scrollowania: Co ciekawe, sam fizyczny akt przewinięcia ekranu (Brainrot Scroll i Smart Scroll) wyrzuca wskaźnik wysoko na plus. Oznacza to, że układ nagrody w mózgu wciąż dostaje krótkotrwały, silny "strzał" dopaminowy i chęć na nową treść. W trybie Brainrot tworzy to błędne koło: ogólny stan jest nieprzyjemny/znużony, ale mózg wykonuje scroll, by poczuć chwilową ulgę i stymulację.

#### Kora Ruchowa (C4 vs C3) – Mechanika ruchu
W EEG spadek fali Alfa oznacza wzrost aktywności mózgu. Dodatni wskaźnik asymetrii w tym miejscu oznacza, że lewa kora ruchowa pracowała mocniej niż prawa.

Widać potężny, równy skok na plus dla samego aktu przewijania ("Brainrot Scroll" i "Smart Scroll"). To aktywacja lewej półkuli, kontrolującej prawą rękę/kciuk uczestnika obsługujący telefon.

#### Kora Ciemieniowa (P4 vs P3) i Potyliczna (O2 vs O1) – Przetwarzanie sensoryczne
We wszystkich trybach te wskaźniki są na plusie. Oznacza to, że w tylnych częściach mózgu uczestnika stale dominuje lewa półkula.

Lewa strona odpowiada za analizę języka, logikę i sekwencyjne przetwarzanie obrazu. Wyniki te wyraźnie sugerują, że użytkownik, niezależnie od trybu, nie tylko gapił się na obrazki, ale jego mózg ciągle dekodował język pisany (czytał opisy, napisy na filmach, komentarze).

Mózg tej osoby doskonale ukazuje pułapkę, jaką jest "doomscrolling" (Brainrot). W ujęciu ogólnym, szybkie i bezmyślne przewijanie treści wywołuje u niej silny stres poznawczy lub znużenie (wyraźne wycofanie emocjonalne z przodu głowy).

## Scroll

In [ ]:
PROJECT_ROOT = Path.cwd().parent 

USER_MAP_PATH = PROJECT_ROOT / "EEG" / "EEG" / "user_map.json"
INTERACTIONS_PATH = PROJECT_ROOT / "EEG" / "research_events_rows.csv"
USERS_PATH = PROJECT_ROOT / "EEG" / "Users_rows.csv"

FILE_NAME = 'Case_02_ExG.csv'

In [ ]:
# Extract the data
scroll_data = extract_scroll_events(FILE_NAME, INTERACTIONS_PATH, USERS_PATH, USER_MAP_PATH)

if scroll_data is not None and not scroll_data.empty:
    # Create MNE Annotations
    new_annotations = mne.Annotations(
        onset=scroll_data['onset'].values,
        duration=scroll_data['duration'].values,
        description=scroll_data['description'].values
    )

    # Overwriting existing annotations to ensure clarity for this analysis
    raw_clean.set_annotations(new_annotations)
    
    print("SUCCESS: Annotations applied to raw_clean.")
    print(f"Time range of scrolls: {scroll_data['onset'].min():.2f}s to {scroll_data['onset'].max():.2f}s")
else:
    print("WARNING: No scroll data found. Plots will be empty.")

In [ ]:
# --- ANALYSIS: MOTOR CORTEX (BETA) ---
# We look for "Beta Rebound" on channel C3.
# Ensure raw_clean has annotations applied (Cell 2) before running this.

compare_stages(
    raw_clean, 
    band='Beta', 
    fmin=13, 
    fmax=30, 
    channel='C3',  
    duration=DURATION
)

In [ ]:
# REWARD/MOTIVATION SYSTEM (Frontal Alpha)
# Channel: F3 (Left Frontal)
# Band: Alpha (8-12 Hz)
# Interpretation: THE LOWER THE GRAPH, THE GREATER THE “REWARD/ENGAGEMENT.”

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='F3',   # Key channel for positive/aspirational emotions
    duration=DURATION
)

In [ ]:
# RELAXATION (Alpha)
# Channel: O1 (Back of the head)
# Band: Alpha (8-12 Hz)
# Interpretation: HIGH LINE = RELAXATION / ZOMBIE MODE. LOW = WORK.

# P3 P4 F3 F4

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='O1', 
    duration=DURATION
)

In [ ]:
# INTELLECTUAL EFFORT (Frontal Theta) ---
# Channel: Fz (Center of forehead)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = STRONG THINKING/MEMORIZATION.

target_channel = 'Fz'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# COGNITIVE CONTROL & INHIBITION (Right Frontal Theta) ---
# Channel: F4 (Right forehead / Right Frontal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = SUSTAINED FOCUS & SUPPRESSING DISTRACTIONS (Mental effort to stay on task or regulate emotional impulses).

target_channel = 'F4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VERBAL & ANALYTICAL MEMORY (Left Parietal Theta) ---
# Channel: P3 (Left back of head / Left Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = PROCESSING TEXT/LANGUAGE (Reading captions, understanding speech, verbal working memory).

target_channel = 'P3'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VISUOSPATIAL PROCESSING & ATTENTION (Right Parietal Theta) ---
# Channel: P4 (Right back of head / Right Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = ENCODING VISUAL/SPATIAL INFO (Processing layout, movement, spatial memory, or complex imagery).

target_channel = 'P4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

#### Kora ruchowa (Akcja fizyczna) – Pasmo Beta (13-30 Hz) na C3
Elektroda C3 bada korę odpowiadającą za ruch prawej dłoni.

**Smart Scroll**: Przy rzadszych scrollach, widać, wzorzec zjawiska ERD/ERS. Tuż przed linią i na samej linii przewijania następuje spadek linii trendu (przygotowanie i ruch palca, co wycisza betę), a chwuilę po scrollu występuje tzw. "rebound" (odbicie) – mocny pik w górę.

**Brainrot Scroll**: Ponieważ ruchy palcem są ciągłe, a odstępy między czerwonymi liniami bardzo małe, kora ruchowa nie ma czasu na pełne "odbicie" po zakończeniu gestu. Wykres fal Beta drży w ciągłym, chaotycznym napięciu, co pokazuje, że układ motoryczny jest zmuszany do stałej gotowości.

#### Kora potyliczna (Rejestracja nowego obrazu) – Pasmo Alpha (8-12 Hz) na O1
Wyciszenie (spadek mocy) fal Alpha z tyłu głowy oznacza, że mózg intensywnie wpatruje się w nową informację wizualną.

**Smart Scroll**: Prawie po każdym scrollu linia trendu fali Alpha "nurkuje" w dół. Po przewinięciu ekranu pojawia się całkowicie nowa treść, kora wzrokowa gwałtownie się aktywuje do jej zdekodowania (Alpha spada). Następnie linia delikatnie faluje lub lekko rośnie, gdy użytkownik przez dłuższą chwilę zapoznaje się ze stabilnym obrazem.

**Brainrot Scroll**: Tutaj sygnał Alpha skacze w niezwykle szybkim, poszarpanym rytmie. Ciągłe miganie nowych filmików nie pozwala korze wzrokowej wejść w żaden stabilny etap analizy. Fale na zmianę uderzają o dno i wystrzeliwują w górę, co często bywa mechanizmem "odcinania się" uwagi od nadmiaru bodźców wizualnych.

#### Kora czołowa i ciemieniowa (Obciążenie poznawcze i uwaga) – Pasmo Theta (4-8 Hz) na Fz, F4 i P3/P4
To jest moment, w którym Twoje czyszczenie danych zagrało główną rolę! Wcześniej z przodu głowy widzieliśmy gigantyczne piki – w dużej mierze wywołane po prostu mruganiem rzęs. Teraz mamy prawdziwy obraz obciążenia umysłowego (cognitive load).

**Smart Scroll**: Na wykresach dla Fz i F4 dla trybu Smart widać wyraźnie, że chwilę po wykonaniu przewinięcia występuje pik. Zatem w trybie Smart zachodzi naturalny łańcuch: ruch -> nowa treść -> płat czołowy włącza zaczyna pracować, żeby to przetworzyć i zrozumieć (skok Theta) -> obciążenie spada, gdy proces się kończy.

**Brainrot Scroll**: Płaty czołowe są "zalewane". Na wykresach Brainrot dla Fz i F4 piki są nieustanne i poszarpane. Mózg jest "przebodźcowany". Przez to, że scrolle następują jeden po drugim, kora czołowa i pamięć robocza nie potrafią dokończyć analizy jednego bodźca, a już są zasypywane kolejnym. Wykres Thety nie przypomina ułożonych pagórków reakcji, tylko jedno wielkie pasmo wysokiego obciążenia.